In [14]:
# Run using ms-toolsai.jupyter for VSCode

In [ ]:
# Install dependencies
%pip install anthropic python-dotenv

In [ ]:
#  Load env vars
from dotenv import load_dotenv
# import os

load_dotenv()
# print(os.environ['ANTHROPIC_API_KEY'])

In [17]:
# Create API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [18]:
# Helper functions to preserve history
def add_user_message(messages, text):
  user_message = {"role": "user", "content": text}
  messages.append(user_message)

def add_assistant_message(messages, text):
  assistant_message = {"role": "assistant", "content": text}
  messages.append(assistant_message)

def chat(messages, system=None, temperature=0.5, stop_sequences=[]):
  params = {
    "model": model,
    "max_tokens": 1000,
    "messages": messages,
    "temperature": temperature,
    "stop_sequences": stop_sequences,
  }
  
  if system:
    params["system"] = system
  
  message = client.messages.create(**params)
  return message.content[0].text

In [ ]:
import json

def generate_dataset():
  """"Generates a dataset for a prompt evaluation"""

  prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "json" or "python" or "regex"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

  messages = []
  add_user_message(messages, prompt)
  add_assistant_message(messages, "```json")
  text = chat(messages, stop_sequences=["```"])
  return json.loads(text)

In [ ]:
dataset = generate_dataset()

with open('dataset.json', 'w') as f:
  json.dump(dataset, f, indent=2)

In [ ]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    
    # THIS is the meat of it. The actual PROMPT we want to evaluate.
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON or a plain Regex
* Do not add any comments or commentary explanation
"""
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [22]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
  eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
  "strengths": string[],
  "weaknesses": string[],
  "reasoning": string,
  "score": number
}}
"""

  messages = []
  add_user_message(messages, eval_prompt)
  add_assistant_message(messages, "```json")
  eval_text = chat(messages, stop_sequences=["```"])
  return json.loads(eval_text)

In [23]:
# Functions to validate the output structure
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(test_case, response):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [24]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    
    output = run_prompt(test_case)
    model_grade = grade_by_model(test_case, output)
    
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    syntax_score = grade_syntax(test_case, output)

    score = (model_score + syntax_score) / 2

    return {
        "test_case": test_case,
        "output": output,
        "score": score,
        "reasoning": reasoning
    }

In [25]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [26]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 6.333333333333333


In [27]:
print(json.dumps(results, indent=2))

[
  {
    "test_case": {
      "task": "Parse an AWS CloudFormation template and extract all resource logical IDs",
      "format": "python"
    },
    "output": "\nimport json\nimport sys\n\ndef extract_resource_ids(template_str):\n    \"\"\"Extract all resource logical IDs from a CloudFormation template.\"\"\"\n    try:\n        template = json.loads(template_str)\n    except json.JSONDecodeError:\n        return []\n    \n    resources = template.get('Resources', {})\n    return list(resources.keys())\n\nif __name__ == \"__main__\":\n    template_input = sys.stdin.read()\n    resource_ids = extract_resource_ids(template_input)\n    print(json.dumps(resource_ids, indent=2))\n",
    "score": 8.0,
    "reasoning": "The solution correctly solves the stated task for JSON CloudFormation templates but has limited practical utility since AWS CloudFormation templates are frequently written in YAML. The error handling is present but minimal, and there's no validation of template correctness b